# RAG DAY 4 — Multi-Document Support
# Goal: Upload multiple PDFs, switch between them,
#       search across all or one specific document
# Run each section in Jupyter Notebook cell by cell

In [9]:
import os
import uuid#It is used to generate IDs that are extremely unlikely to repeat.
from datetime import datetime
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq
from fpdf import FPDF#helps in creatiing pdfs

load_dotenv()
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your-groq-key-here")

# Create Multiple Test PDFs

In [10]:
print("=" * 55)
print("PART 2: Creating Test Documents")
print("=" * 55)

def create_test_pdf(filename, content_pages):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    for content in content_pages:
        pdf.add_page()
        pdf.set_font("Arial", size=12)
        pdf.multi_cell(0, 10, content)
    pdf.output(filename)
    print(f"Created: {filename}")

# Document 1 — ML textbook
create_test_pdf("ml_textbook.pdf", [
    """Introduction to Machine Learning

Machine learning is a branch of artificial intelligence that enables
computers to learn from data without being explicitly programmed.
The field was pioneered by Arthur Samuel in 1959.

There are three main paradigms: supervised learning where models learn
from labeled examples, unsupervised learning where models find hidden
patterns, and reinforcement learning where agents learn from rewards.""",

    """Deep Learning and Neural Networks

Deep learning uses neural networks with multiple layers. CNNs excel
at image recognition. RNNs handle sequential data. Transformers have
revolutionized NLP since 2017.

Applications include image classification, speech recognition,
natural language processing, and autonomous vehicles.""",

    """Applications of Machine Learning

In healthcare ML models detect diseases from medical images.
In finance algorithms detect fraudulent transactions in real time.
Recommendation systems use collaborative filtering to suggest content.
Self driving cars use computer vision and reinforcement learning."""
])

# Document 2 — Python guide
create_test_pdf("python_guide.pdf", [
    """Python Programming Fundamentals

Python is a high-level language created by Guido van Rossum in 1991.
It emphasizes readability and simplicity. Python uses indentation
instead of curly braces. Variables are dynamically typed.""",

    """Python for Data Science

Key libraries include NumPy for numerical computing, Pandas for data
manipulation, and Matplotlib for visualization. Scikit-learn provides
machine learning tools. PyTorch and TensorFlow are deep learning
frameworks. Jupyter Notebooks allow interactive development.""",

    """Python Best Practices

Follow PEP 8 style guide. Functions should do one thing well.
Use meaningful variable names. Write docstrings for all functions.
Use virtual environments to isolate dependencies. Always pin
dependencies in requirements.txt for reproducibility."""
])



print("\nCreated 2 test documents:")
print("  1. ml_textbook.pdf  — 3 pages")
print("  2. python_guide.pdf — 3 pages")


PART 2: Creating Test Documents
Created: ml_textbook.pdf
Created: python_guide.pdf

Created 2 test documents:
  1. ml_textbook.pdf  — 3 pages
  2. python_guide.pdf — 3 pages


# MultiDocumentRAG Class

In [ ]:
class MultiDocumentRAG:
    """
    RAG system supporting multiple documents.

    Design:
    - ONE ChromaDB collection for ALL documents
    - Each chunk stores doc_id in metadata
    - Filter by doc_id to search one document
    - No filter to search all documents
    """
    def __init__(self):
        self.embedder=SentenceTransformer("all-MiniLM-L6-v2")

        self.client=chromadb.PersistentClient(path="./multi_doc_db")
        try:
            self.client.delete_collection("all_documents")
        except:
            pass
        self.collection=self.client.create_collection("all_documents")


        # Track uploaded documents
        self.documents = {}     # { doc_id: info_dict }
        self.histories = {}     # { doc_id: [messages] }
        self.global_history = []

        self.groq=Groq(api_key="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")


    def add_document(self,pdf_path,doc_name=None):
        """Add a PDF. Returns doc_id."""
        doc_id=str(uuid.uuid4())[:8]
        doc_name=doc_name or os.path.basename(pdf_path)

        print(f"\nAdding: {doc_name} (ID: {doc_id})")
        loader   = PyPDFLoader(pdf_path)
        pages    = loader.load()
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=50
        )
        chunks = splitter.split_documents(pages)

        texts      = [c.page_content for c in chunks]
        embeddings = self.embedder.encode(texts)

        #store doc_id in every chunk's metadata
        self.collection.add(
            documents=texts,
            embeddings=embeddings.tolist(),
            metadatas=[#earlier oly page number now additional doc_id,doc_name
                {
                    "doc_id":doc_id,
                    "doc_name":doc_name,
                    "page":c.metadata.get("page",0)+1
                }
                for c in chunks
            ],
            ids = [f"{doc_id}_{i}" for i in range(len(chunks))]
        )#Ex (id)abc123_0 ->(metadata){"doc_id":"abc123","doc_name":"AI.pdf","page":1}

        # Register
        self.documents[doc_id]={
            "name":doc_name,
            "pages":len(pages),
            "chunks":len(chunks),
            "added":datetime.now().strftime("%H:%M")
        }

        self.histories[doc_id]=[]
        print(f"✅ {doc_name}: {len(pages)} pages, {len(chunks)} chunks")
        return doc_id
    
    def list_documents(self):
        """Show all documents"""
        print("\nUploaded Documents:")
        for doc_id,info in self.documents.items():
            print(f"[{doc_id}] {info['name']}")
            print(f"  Pages: {info['pages']} | Chunks: {info['chunks']} | Added: {info['added']}")

    def remove_document(self,doc_id):
        """Remove a document and all its chunks"""
        if doc_id not in self.documents:
            print(f"Document {doc_id} not found")
            return
        name=self.documents[doc_id]["name"]
        results=self.collection.get(where={"doc_id":doc_id})
        if results["ids"]:
            self.collection.delete(ids=results["ids"])
        del self.documents[doc_id]
        del self.histories[doc_id]
        print(f"✅ Removed: {name}")

    def retrieve(self,question,doc_id=None,n=3):
            """
            Retrieve chunks.
            doc_id given → search that document only
            doc_id None  → search all documents
            """
            q_emb = self.embedder.encode([question]).tolist()
            # 🔑 where filter restricts search to one document
            query_args = dict(
            query_embeddings = q_emb,
            n_results        = n,
            include          = ["documents", "metadatas", "distances"]
            )
            if doc_id:#Only search chunks whose metadata has:doc_id = abc123
                query_args["where"] = {"doc_id": doc_id}

            results = self.collection.query(**query_args)#This unpacks dictionary into keyword arguments query_args = {"n_results":3}->query_args(n_results=3)

            return [
            {
                "text":      results["documents"][0][i],
                "page":      results["metadatas"][0][i]["page"],
                "doc_name":  results["metadatas"][0][i]["doc_name"],
                "doc_id":    results["metadatas"][0][i]["doc_id"],
                "relevance": round(1 - results["distances"][0][i], 2)
            }
                for i in range(len(results["documents"][0]))
            ]
    
    def chat(self, question, doc_id=None):
        """
        Ask a question with conversation memory.
        doc_id=None → global search across all docs
        doc_id=xyz  → search within that doc only
        """
        #each document has its own conversation history
        history = self.histories.get(doc_id, []) if doc_id \
                  else self.global_history

        chunks  = self.retrieve(question, doc_id)
        context = "\n\n".join(
            f"[{c['doc_name']} — Page {c['page']}]: {c['text']}"
            for c in chunks
        )
        sources = [
            {"doc": c["doc_name"], "page": c["page"],
             "relevance": c["relevance"]}
            for c in chunks
        ]

        system_msg = {
            "role": "system",
            "content": f"""You are a helpful assistant searching {scope}.
Answer using ONLY the context. Cite as (DocumentName, Page X).
Say "I don't have that information" if context is insufficient.

Context:
{context}"""
        }

        messages = [system_msg] + history + [
            {"role": "user", "content": question}
        ]

        response = self.groq.chat.completions.create(
            model       = "llama-3.3-70b-versatile",
            messages    = messages,
            max_tokens  = 400,
            temperature = 0.1
        )
        answer = response.choices[0].message.content

        # Update history (trim to last 10)
        updated = history + [
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer}
        ]
        updated = updated[-10:]

        if doc_id:
            self.histories[doc_id] = updated
        else:
            self.global_history = updated

        return {"answer": answer, "sources": sources,
                "mode": "single" if doc_id else "global"}


    def compare(self, question, doc_id_1, doc_id_2):
        """Compare what two documents say about a topic"""
        name1   = self.documents[doc_id_1]["name"]
        name2   = self.documents[doc_id_2]["name"]
        chunks1 = self.retrieve(question, doc_id_1, n=2)
        chunks2 = self.retrieve(question, doc_id_2, n=2)

        ctx1 = "\n".join(f"[Page {c['page']}]: {c['text']}" for c in chunks1)
        ctx2 = "\n".join(f"[Page {c['page']}]: {c['text']}" for c in chunks2)

        prompt = f"""Compare what these two documents say about: "{question}"

{name1}:
{ctx1}

{name2}:
{ctx2}

Structure your answer as:
1. {name1} says...
2. {name2} says...
3. Key similarities...
4. Key differences..."""

        response = self.groq.chat.completions.create(
            model      = "llama-3.3-70b-versatile",
            messages   = [{"role": "user", "content": prompt}],
            max_tokens = 600,
            temperature= 0.1
        )
        return response.choices[0].message.content


    def reset(self, doc_id=None):
        """Reset chat history"""
        if doc_id:
            self.histories[doc_id] = []
            print(f"Reset history for {self.documents[doc_id]['name']}")
        else:
            self.global_history = []
            print("Reset global history")

# test

In [12]:
print("\n" + "=" * 55)
print("PART 4: Testing")
print("=" * 55)

rag = MultiDocumentRAG()

# Add documents
doc1 = rag.add_document("ml_textbook.pdf",  "ML Textbook")
doc2 = rag.add_document("python_guide.pdf", "Python Guide")

rag.list_documents()

# Test 1: Single document search
print("\nTEST 1: Within ML Textbook only")
r = rag.chat("What are the types of machine learning?", doc_id=doc1)
print(f"A: {r['answer']}")
print(f"Sources: {r['sources']}")

# Test 2: Different document
print("\nTEST 2: Within Python Guide only")
r = rag.chat("What libraries are used for data science?", doc_id=doc2)
print(f"A: {r['answer']}")
print(f"Sources: {r['sources']}")

# Test 3: Global search
print("\nTEST 3: Across ALL documents")
r = rag.chat("What skills do I need to learn AI?")
print(f"A: {r['answer']}")
print(f"Sources: {r['sources']}")

# Test 4: Follow-up conversation
print("\nTEST 4: Follow-up within ML Textbook")
r1 = rag.chat("What is deep learning?", doc_id=doc1)
print(f"Q1: What is deep learning?")
print(f"A1: {r1['answer'][:150]}...")
r2 = rag.chat("Give me examples of it", doc_id=doc1)
print(f"\nQ2: Give me examples of it")
print(f"A2: {r2['answer'][:150]}...")

# Test 5: Compare documents
print("\nTEST 5: Compare ML Textbook vs Python Guide")
comparison = rag.compare(
    "What Python tools or libraries are mentioned?",
    doc1, doc2
)
print(comparison)



PART 4: Testing


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6693.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Adding: ML Textbook (ID: 5ca74426)
✅ ML Textbook: 3 pages, 3 chunks

Adding: Python Guide (ID: 1bb37136)
✅ Python Guide: 3 pages, 3 chunks

Uploaded Documents:
[5ca74426] ML Textbook
  Pages: 3 | Chunks: 3 | Added: 10:26
[1bb37136] Python Guide
  Pages: 3 | Chunks: 3 | Added: 10:26

TEST 1: Within ML Textbook only
A: There are three main paradigms of machine learning: 
1. Supervised learning, where models learn from labeled examples, 
2. Unsupervised learning, where models find hidden patterns, and 
3. Reinforcement learning, where agents learn from rewards (ML Textbook, Page 1).
Sources: [{'doc': 'ML Textbook', 'page': 1, 'relevance': 0.36}, {'doc': 'ML Textbook', 'page': 3, 'relevance': 0.1}, {'doc': 'ML Textbook', 'page': 2, 'relevance': -0.19}]

TEST 2: Within Python Guide only
A: Key libraries for data science include NumPy for numerical computing, Pandas for data manipulation, and Matplotlib for visualization. Additionally, Scikit-learn provides machine learning tools, and PyTor